In [4]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
from PIL import Image

for env_id in gym.registry.keys():
    if "PickCube" in env_id:
        print("  ", env_id)

ENV_ID = "PickCube-v1"
env = gym.make(
    ENV_ID,
    obs_mode="rgb",
    control_mode="pd_ee_delta_pose",
    render_mode="rgb_array",
    sensor_configs=dict(width=224, height=224),
)

obs, info = env.reset(seed=0)
for key in obs:
    value = obs[key]
    if isinstance(value, dict):
        for sub_key in value:
            inner = value[sub_key]
            if isinstance(inner, dict):
                for leaf in inner:
                    print(f"   {key}/{sub_key}/{leaf}: {getattr(inner[leaf], 'shape', '')}")
            else:
                    print(f"   {key}/{sub_key}: {getattr(inner, 'shape', '')}")
    else:
        print(f"   {key}: {getattr(value, 'shape', type(value).__name__)}")
print("\n info 키:", list(info.keys()))

frame = env.render().cpu().numpy()
if frame.ndim == 4:
    frame = frame[0]
print("frame shape:", frame.shape)
Image.fromarray(frame.astype(np.uint8)).save("outputs/env_check.png")
print("저장 완료: outputs/env_check.png")

env.close()

   PickCube-v1
   PickCubeSO100-v1
   PickCubeWidowXAI-v1
   TwoRobotPickCube-v1
   agent/qpos: torch.Size([1, 9])
   agent/qvel: torch.Size([1, 9])
   extra/is_grasped: torch.Size([1])
   extra/tcp_pose: torch.Size([1, 7])
   extra/goal_pos: torch.Size([1, 3])
   sensor_param/base_camera/extrinsic_cv: torch.Size([1, 3, 4])
   sensor_param/base_camera/cam2world_gl: torch.Size([1, 4, 4])
   sensor_param/base_camera/intrinsic_cv: torch.Size([1, 3, 3])
   sensor_data/base_camera/rgb: torch.Size([1, 224, 224, 3])

 info 키: ['elapsed_steps', 'success', 'is_obj_placed', 'is_robot_static', 'is_grasped', 'reconfigure']
frame shape: (512, 512, 3)
저장 완료: outputs/env_check.png


In [1]:
"""
실습 5: 기성 해법으로 성공률 상한을 확인 (루프·환경·판정 검증)

ManiSkill 내장 motion planning 해법은 이 환경에서 쓸 수 없다. mani_skill 3.0.1 이 핀으로 박은
mplib==0.1.1 바이너리가 numpy 1.x C API 로 빌드돼 있어 numpy 2.5.1 과 ABI 가 안 맞고,
mplib.Planner 생성 시점에 세그폴트로 죽는다 (numpy 를 1.26 으로 내리면 scipy/opencv 가 깨진다).
그래서 README §8 이 함께 허용하는 다른 기성 해법 — scripted 정책 — 으로 상한을 잡는다.

이 scripted 정책은 카메라를 보지 않고 큐브·목표 좌표를 시뮬레이터에서 직접 읽는다 (인식 문제를 우회).
motion planning 과 달리 실습 6 과 **같은 control_mode(pd_ee_delta_pose)** 를 쓰므로
정규화 action 경로까지 함께 검증된다. 단 OpenVLA 출력을 변환하는 레이어는 지나가지 않으므로
변환 레이어는 이 검증으로 배제되지 않는다.
"""
import json                                    # 결과를 원시 형태로 저장하기 위해
import numpy as np
import gymnasium as gym
import mani_skill.envs


ENV_ID = "PickCube-v1"                         # <- 실습 2 확정값
MAX_EPISODE_STEPS = 200                        # <- 실습 4-3 확정값 (실습 6 과 동일해야 한다)
SEEDS = list(range(20))                        # <- 실습 6 과 공유할 고정 목록
POS_LIMIT = 0.1                                # pd_ee_delta_pose 의 위치 한계 (m). action 1.0 = 0.1 m
MAX_STEP_M = 0.03                              # 한 step 에 요청할 최대 이동량 (m). 크게 잡으면 PD 추종이 흔들린다
APPROACH_HEIGHT = 0.05                         # 큐브 위 어느 높이에서 하강을 시작할지 (m)
CLOSE_STEPS = 8                                # 그리퍼가 실제로 닫히기까지 기다리는 step 수


def to_vec(pose_field):
    """배치 텐서로 오는 좌표를 (3,) numpy 벡터로 바꾼다.

    Args:
        pose_field: `pose.p` 같은 (1, 3) 형태의 GPU 텐서

    Returns:
        (3,) float numpy 배열
    """
    return np.asarray(pose_field.cpu())[0]     # GPU -> CPU -> numpy, 배치 차원 제거


def run_episode(env, seed):
    """scripted 정책으로 한 episode 를 수행한다.

    좌표를 직접 읽어 접근 -> 하강 -> 파지 -> 목표 이동 -> 정지 순서로 진행한다.
    회전 델타는 전부 0 이다 — Panda 의 초기 자세가 이미 그리퍼를 아래로 향하고 있어
    위치 제어만으로 큐브를 잡을 수 있다.

    Args:
        env: `pd_ee_delta_pose` 로 생성된 ManiSkill 환경
        seed: 초기 배치를 재현하는 정수

    Returns:
        (성공 여부, 소비한 env step 수, 마지막 info 의 부분 조건 딕셔너리)
    """
    base = env.unwrapped                       # 큐브·목표 좌표는 wrapper 를 벗겨야 보인다
    obs, info = env.reset(seed=seed)            # 고정 seed 로 초기 배치 재현
    phase = "above"                            # 현재 단계 (above -> descend -> close -> lift -> hold)
    close_count = 0                            # 그리퍼 닫기 명령을 몇 step 유지했는지

    for step in range(MAX_EPISODE_STEPS):
        tcp = to_vec(base.agent.tcp.pose.p)    # 그리퍼 끝(TCP) 현재 위치
        cube = to_vec(base.cube.pose.p)        # 큐브 현재 위치 (특권 정보 — 카메라를 안 본다)
        goal = to_vec(base.goal_site.pose.p)   # 목표 지점 위치
        grip = 1.0                             # 기본은 열림 (+1 = 열림, 계약 표 5번)

        if phase == "above":                   # 1) 큐브 위쪽으로 이동
            target = cube + np.array([0.0, 0.0, APPROACH_HEIGHT])
            if np.linalg.norm(target - tcp) < 0.008:      # 충분히 도달하면 다음 단계
                phase = "descend"
        elif phase == "descend":               # 2) 큐브 중심 높이까지 하강
            target = cube
            if np.linalg.norm(target - tcp) < 0.006:
                phase = "close"
        elif phase == "close":                 # 3) 제자리에서 그리퍼 닫기
            target = tcp                       # 이동 없음 (델타 0)
            grip = -1.0                        # -1 = 닫힘
            close_count += 1
            if close_count >= CLOSE_STEPS:
                phase = "lift"
        elif phase == "lift":                  # 4) 큐브를 든 채 목표 지점으로 이동
            target = goal
            grip = -1.0
            if np.linalg.norm(goal - cube) < 0.02:        # goal_thresh(0.025) 보다 보수적으로
                phase = "hold"
        else:                                  # 5) 정지 — success 는 is_robot_static 도 요구한다
            target = tcp
            grip = -1.0

        delta = np.clip(target - tcp, -MAX_STEP_M, MAX_STEP_M)     # step당 이동량 제한
        action = np.concatenate([
            delta / POS_LIMIT,                 # 미터 -> [-1, 1] 정규화 (계약 표 1번과 같은 규칙)
            np.zeros(3),                       # 회전 델타 없음
            [grip],                            # gripper 명령
        ]).astype(np.float32)

        obs, reward, terminated, truncated, info = env.step(action)
        if bool(info["success"].item()):       # 성공 즉시 종료
            return True, step + 1, info
        if terminated or truncated:            # 환경이 스스로 끝냈으면 종료
            break

    return False, step + 1, info               # step 예산을 다 쓰고 실패


print("=" * 60)
print("실습 5: 상한 대조 (scripted 정책)")
print("=" * 60)


# -- 5-1. 실습 6 과 같은 조건으로 환경 생성 --
env = gym.make(
    ENV_ID,
    obs_mode="rgb",                            # 이 해법은 이미지를 안 쓰지만 조건을 맞춰 둔다
    control_mode="pd_ee_delta_pose",           # 실습 6 과 동일 — 정규화 action 경로까지 검증된다
    render_mode="rgb_array",                   # 헤드리스
    sensor_configs=dict(width=224, height=224),   # 실습 6 과 동일한 관측 카메라 설정
    max_episode_steps=MAX_EPISODE_STEPS,       # 실습 6 과 같은 env step 예산
)


# -- 5-2. 20 episode 실행 --
print("\n[5-2] scripted 해법 20 episode")
success_count = 0                              # 성공 episode 수
records = []                                   # episode 별 결과
for seed in SEEDS:
    solved, steps, info = run_episode(env, seed)
    stages = {                                 # 실패한 경우 어디까지 갔는지 남긴다
        "is_grasped": bool(info["is_grasped"].item()),
        "is_obj_placed": bool(info["is_obj_placed"].item()),
        "is_robot_static": bool(info["is_robot_static"].item()),
    }
    success_count += int(solved)               # 성공이면 1 누적
    records.append({"seed": seed, "solved": solved, "steps": steps, **stages})
    print(f"   seed{seed:02d}: solved={solved} steps={steps} grasped={stages['is_grasped']}")


print(f"\n상한 성공률: {success_count}/{len(SEEDS)}")
steps_used = [record["steps"] for record in records]
print(f"소비 step: 최소 {min(steps_used)} / 최대 {max(steps_used)} / 평균 {sum(steps_used) / len(steps_used):.1f}"
      f" (실습 6 예산 {MAX_EPISODE_STEPS})")


# -- 5-3. 하한 대조 — 아무것도 하지 않는 정책의 성공률 --
# PickCube 는 목표 지점을 무작위로 뽑으므로, 목표가 큐브 초기 위치에서 goal_thresh 안에
# 떨어지는 seed 가 섞인다. 그 seed 는 조작 없이도 success 가 되어 성공률을 공짜로 올린다.
# 실습 6 의 성공률을 해석하려면 이 공짜분을 먼저 알아야 한다.
print("\n[5-3] 무행동 정책 20 episode (하한)")
zero_action = np.zeros(7, dtype=np.float32)    # 전 차원 0 = 이동·회전 없음
noop_hits = []                                 # 무행동으로 성공한 seed 목록
for seed in SEEDS:
    obs, info = env.reset(seed=seed)
    for step in range(MAX_EPISODE_STEPS):
        obs, reward, terminated, truncated, info = env.step(zero_action)
        if bool(info["success"].item()):       # 조작 없이 성공 -> 애초에 목표 안에 있던 배치
            noop_hits.append(seed)
            break
        if terminated or truncated:
            break
print(f"   무행동 성공 seed: {noop_hits} -> {len(noop_hits)}/{len(SEEDS)}")


# -- 5-4. 원시 결과 저장 (outputs/harness_check.md 작성의 근거) --
with open("outputs/harness_check.json", "w") as f:
    json.dump({"env_id": ENV_ID, "control_mode": "pd_ee_delta_pose",
               "max_episode_steps": MAX_EPISODE_STEPS, "solver": "scripted (privileged state)",
               "seeds": SEEDS, "upper_bound": success_count, "records": records,
               "noop_lower_bound": len(noop_hits), "noop_success_seeds": noop_hits}, f, indent=2)
print("저장 완료: outputs/harness_check.json")


env.close()

/workspace/study/physical-ai-study/Studies/Phase 4.5/.venv-sim/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/study/physical-ai-study/Studies/Phase 4.5/.venv-sim/lib/python3.12/site-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(


실습 5: 상한 대조 (scripted 정책)

[5-2] scripted 해법 20 episode
   seed00: solved=True steps=49 grasped=True
   seed01: solved=True steps=44 grasped=True
   seed02: solved=True steps=28 grasped=True
   seed03: solved=True steps=44 grasped=True
   seed04: solved=True steps=48 grasped=True
   seed05: solved=True steps=45 grasped=True
   seed06: solved=True steps=43 grasped=True
   seed07: solved=True steps=42 grasped=True
   seed08: solved=True steps=9 grasped=False
   seed09: solved=True steps=49 grasped=True
   seed10: solved=True steps=48 grasped=True
   seed11: solved=True steps=32 grasped=True
   seed12: solved=True steps=31 grasped=True
   seed13: solved=True steps=47 grasped=True
   seed14: solved=True steps=31 grasped=True
   seed15: solved=True steps=38 grasped=True
   seed16: solved=True steps=42 grasped=True
   seed17: solved=True steps=42 grasped=True
   seed18: solved=True steps=36 grasped=True
   seed19: solved=True steps=40 grasped=True

상한 성공률: 20/20
소비 step: 최소 9 / 최대 49 / 평균 39

In [7]:
import inspect
import gymnasium as gym
import mani_skill.envs

ENV_ID = "PickCube-v1"

env = gym.make(ENV_ID, obs_mode="rgb", control_mode="pd_ee_delta_pose")

print(env.action_space)
print("low: ", env.action_space.low)
print("high: ", env.action_space.high)

print(env.unwrapped.agent.supported_control_modes)
print(inspect.getsource(type(env.unwrapped).evaluate))
print([name for name in dir(env.unwrapped) if not name.startswith("_")])

STEP_CAP = 100
for episode in range(20):
    obs, info = env.reset(seed=episode)
    success = False
    for step in range(STEP_CAP):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, infor = env.step(action)
        if info.get("success", False):
            success = True
            break
        if terminated or truncated:
            break
    print(f"epp{episode:02d}: success={success} steps={step + 1}")

env.close()

Box(-1.0, 1.0, (7,), float32)
low:  [-1. -1. -1. -1. -1. -1. -1.]
high:  [1. 1. 1. 1. 1. 1. 1.]
['pd_joint_delta_pos', 'pd_joint_pos', 'pd_ee_delta_pos', 'pd_ee_delta_pose', 'pd_ee_pose', 'pd_joint_target_delta_pos', 'pd_ee_target_delta_pos', 'pd_ee_target_delta_pose', 'pd_joint_vel', 'pd_joint_pos_vel', 'pd_joint_delta_pos_vel']
    def evaluate(self):
        is_obj_placed = (
            torch.linalg.norm(self.goal_site.pose.p - self.cube.pose.p, axis=1)
            <= self.goal_thresh
        )
        is_grasped = self.agent.is_grasping(self.cube)
        is_robot_static = self.agent.is_static(0.2)
        return {
            "success": is_obj_placed & is_robot_static,
            "is_obj_placed": is_obj_placed,
            "is_robot_static": is_robot_static,
            "is_grasped": is_grasped,
        }

['SUPPORTED_OBS_MODES', 'SUPPORTED_RENDER_MODES', 'SUPPORTED_REWARD_MODES', 'SUPPORTED_ROBOTS', 'action_space', 'add_to_state_dict_registry', 'agent', 'backend', 'capture_sens

In [1]:
"""
실습 6: OpenVLA zero-shot baseline (N=20, 고정 seed, 부분 도달률 병기)
"""
import json                                    # 결과를 원시 형태로 저장하기 위해
import numpy as np
import torch
import gymnasium as gym
import mani_skill.envs
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig


ENV_ID = "PickCube-v1"                         # <- 실습 2 확정값
MAX_EPISODE_STEPS = 200                        # <- 실습 4-3: env step 예산 (실습 5 와 동일)
ACTION_REPEAT = 4                              # <- 실습 4-3: 5 Hz 정책을 20 Hz sim 에 맞춘다
POLICY_STEPS = MAX_EPISODE_STEPS // ACTION_REPEAT   # 정책 결정 횟수 = 50
SEEDS = list(range(20))                        # <- 실습 5 와 같은 목록 (변인 고정)
NOOP_SEEDS = [8]                               # <- 실습 5-3 무행동 하한. 이 seed 의 성공은 조작 성공이 아니다
UNNORM_KEY = "bridge_orig"                     # <- 실습 4 에서 근거와 함께 확정한 key
INSTRUCTION = "pick up the cube"               # 영어 단문 고정 (OpenVLA prompt 틀)
POS_LIMIT = 0.1                                # pd_ee_delta_pose 의 위치 한계 (m). action 1.0 = 0.1 m
ROT_SCALE = -0.1                               # 회전 스케일 (rad). 부호가 반전돼 있다 (계약 표 4번)
REACH_DIST = 0.05                              # reached 임계값 (m). 근거는 아래 판정식 주석
LIFT_Z = 0.04                                  # lifted 임계값 (m). 근거는 아래 판정식 주석


def to_maniskill_action(raw_action):
    """OpenVLA 역정규화 출력(7,) 을 ManiSkill pd_ee_delta_pose action(7,) 으로 변환한다.

    변환 규칙의 근거는 `action_contract.md` 의 계약 표에 있다. OpenVLA 는 물리량(m, rad)을
    돌려주고 ManiSkill 은 전 차원 [-1, 1] 정규화값을 받으므로, 이 함수를 거치지 않으면
    위치 명령이 의도의 1/10 로 줄고 회전이 반대로 돈다.

    Args:
        raw_action: `vla.predict_action()` 출력 numpy 배열 (7,).
            [dx,dy,dz](m), [drx,dry,drz](rad), gripper[0,1]

    Returns:
        ManiSkill action (7,) float32. 전 차원 [-1, 1] 정규화값
    """
    pos = raw_action[:3] / POS_LIMIT           # 미터 -> 정규화 (±0.1 m 가 ±1)
    rot = raw_action[3:6] / ROT_SCALE          # 라디안 -> 정규화, rot_lower 곱셈 때문에 부호 반전
    rot_norm = np.linalg.norm(rot)             # 회전은 축별이 아니라 3벡터 노름으로 제한된다
    if rot_norm > 1.0:                         # 노름이 1 을 넘으면 방향을 유지한 채 축소
        rot = rot / rot_norm
    grip = 2.0 * raw_action[6] - 1.0           # [0,1](0=닫힘) -> [-1,1](-1=닫힘)
    action = np.concatenate([np.clip(pos, -1, 1), rot, [np.clip(grip, -1, 1)]])
    return action.astype(np.float32)           # env.step 이 받는 dtype 으로 맞춘다


def to_vec(pose_field):
    """배치 텐서로 오는 좌표를 (3,) numpy 벡터로 바꾼다.

    Args:
        pose_field: `pose.p` 같은 (1, 3) 형태의 GPU 텐서

    Returns:
        (3,) float numpy 배열
    """
    return np.asarray(pose_field.cpu())[0]     # GPU -> CPU -> numpy, 배치 차원 제거


print("=" * 60)
print("실습 6: zero-shot baseline")
print("=" * 60)


# -- 6-1. 모델 로드 (Phase 4 week6 과 동일 설정) --
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)
prompt = f"In: What action should the robot take to {INSTRUCTION}?\nOut:"   # OpenVLA 가 학습된 문장 틀


# -- 6-2. 환경 생성 (실습 5 와 동일 조건 + 관측 카메라 해상도 고정) --
env = gym.make(
    ENV_ID,
    obs_mode="rgb",
    control_mode="pd_ee_delta_pose",           # OpenVLA 출력 형식과 대응 (실습 2)
    render_mode="rgb_array",
    sensor_configs=dict(width=224, height=224),   # 모델 입력 크기와 일치 -> 리사이즈 열화 없음
    max_episode_steps=MAX_EPISODE_STEPS,       # 실습 5 와 같은 env step 예산
)
base = env.unwrapped                           # 큐브·TCP 좌표는 wrapper 를 벗겨야 보인다


# -- 6-3. episode 루프 --
records = []                                   # episode 별 결과를 모은다
grip_raw = []                                  # raw_action[6] 전량. 계약 표 5번의 [0,1] 가정 검증용
for seed in SEEDS:
    obs, info = env.reset(seed=seed)           # 고정 seed 로 초기 배치 재현
    stages = {"reached": False, "grasped": False, "lifted": False, "placed": False}
    done = False                               # 이 episode 를 끝낼지 여부
    for policy_step in range(POLICY_STEPS):    # 정책 결정 50회
        # (a) 관측에서 카메라 이미지 추출 — 키 경로는 실습 2 의 2-3 출력대로
        frame = obs["sensor_data"]["base_camera"]["rgb"]
        frame = np.asarray(frame.cpu() if hasattr(frame, "cpu") else frame)   # GPU 텐서면 내린다
        if frame.ndim == 4:                    # (1, H, W, 3) 이면 첫 장만
            frame = frame[0]
        image = Image.fromarray(frame.astype(np.uint8))       # 이미 224x224 이므로 리사이즈 불필요

        # (b) 추론 — attention_mask 는 넘기지 않는다 (Phase 4 week6 의 크래시 회피)
        inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
        with torch.no_grad():                  # 학습이 아니므로 기울기 계산을 끈다
            raw_action = vla.predict_action(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                unnorm_key=UNNORM_KEY,         # 실습 4 에서 확정
                do_sample=False,               # 결정적 출력 (제어에서는 무작위성 배제)
            )
        grip_raw.append(float(raw_action[6]))  # gripper 차원 원값을 남긴다 (계약 표 5번 검증)

        # (c) 변환 — 실습 4 의 action_contract.md 에 쓴 변환 함수를 그대로 쓴다
        action = to_maniskill_action(raw_action)

        # (d) 실행 — 같은 action 을 ACTION_REPEAT 번 넣어 실효 주기를 5 Hz 로 맞춘다
        for _ in range(ACTION_REPEAT):
            obs, reward, terminated, truncated, info = env.step(action)

            # (e) 부분 도달률 갱신 — 판정 경로는 실습 3 의 3-4 출력에서 확정한 것
            tcp = to_vec(base.agent.tcp.pose.p)               # 그리퍼 끝 현재 위치
            cube = to_vec(base.cube.pose.p)                   # 큐브 현재 위치
            # reached: 실습 5 scripted 정책의 접근 고도(5 cm) 와 같은 값 -> 상한 대조와 같은 척도
            stages["reached"] |= bool(np.linalg.norm(tcp - cube) < REACH_DIST)
            # grasped: 환경이 이미 계산해 info 로 실어 보낸다 -> 판정식을 새로 만들지 않는다
            stages["grasped"] |= bool(info["is_grasped"].item())
            # lifted: 큐브가 모서리로 기울어 서 있을 때 중심 높이가 최대 0.02*sqrt(3)=0.0346 m 이므로,
            #         그보다 위인 0.04 m 를 기준으로 잡으면 "기울어짐"을 "들림"으로 오판하지 않는다
            stages["lifted"] |= bool(cube[2] > LIFT_Z)
            stages["placed"] |= bool(info["success"].item())

            if stages["placed"] or terminated or truncated:   # 성공 또는 환경이 끝냄
                done = True
                break                          # repeat 루프 탈출
        if done:
            break                              # 정책 루프 탈출

    records.append({"seed": seed, "policy_steps": policy_step + 1, **stages})
    print(f"   seed{seed:02d}: {records[-1]}")


env.close()


# -- 6-4. 요약 + 원시 결과 저장 --
print("\n[6-4] 요약")
for stage in ["reached", "grasped", "lifted", "placed"]:
    hit = sum(record[stage] for record in records)          # 해당 단계 도달 episode 수
    print(f"   {stage}: {hit}/{len(SEEDS)}")
# 무행동 하한 seed 의 성공은 조작 능력의 증거가 아니므로 걷어낸 값을 함께 적는다 (실습 5-3)
earned = sum(record["placed"] for record in records if record["seed"] not in NOOP_SEEDS)
print(f"   placed 중 조작으로 얻은 것: {earned}/{len(SEEDS)} (하한 seed {NOOP_SEEDS} 제외)")


# gripper 원값 분포 — 계약 표 5번이 가정한 [0,1] 이산값인지 확인한다
print("\n[6-4] raw_action[6] (gripper) 분포")
print(f"   최소 {min(grip_raw):.3f} / 최대 {max(grip_raw):.3f} / 평균 {sum(grip_raw) / len(grip_raw):.3f}")
print(f"   0.1 미만 {sum(g < 0.1 for g in grip_raw)} / 0.9 초과 {sum(g > 0.9 for g in grip_raw)}"
      f" / 그 사이 {sum(0.1 <= g <= 0.9 for g in grip_raw)} (전체 {len(grip_raw)})")


with open("outputs/zeroshot_baseline.json", "w") as f:
    json.dump({"env_id": ENV_ID, "max_episode_steps": MAX_EPISODE_STEPS,
               "action_repeat": ACTION_REPEAT, "unnorm_key": UNNORM_KEY,
               "reach_dist": REACH_DIST, "lift_z": LIFT_Z, "noop_seeds": NOOP_SEEDS,
               "seeds": SEEDS, "records": records, "gripper_raw": grip_raw}, f, indent=2)
print("저장 완료: outputs/zeroshot_baseline.json")

/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(


실습 6: zero-shot baseline


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  

   seed00: {'seed': 0, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed01: {'seed': 1, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed02: {'seed': 2, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed03: {'seed': 3, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed04: {'seed': 4, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed05: {'seed': 5, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed06: {'seed': 6, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed07: {'seed': 7, 'policy_steps': 50, 'reached': False, 'grasped': False, 'lifted': False, 'placed': False}
   seed08: {'seed': 8, 'policy_steps': 1, 'reached': False, 'grasped': False, 'lifted': False, '